# S4 · AndinaLog 03B · Notebook 2 · Tratamiento de conductores

Este notebook consume las salidas del notebook 1 de `andinalog_hr_drivers.csv` y `catalogo_reglas_tratamiento.csv`. Solo ejecuta reglas cuyo estado sea `APROBADA`. Conserva los seis campos Bronze, registra cada decisión y no obliga a ninguna fila a salir de cuarentena.

Con el catálogo entregado, la única regla aprobada es excluir copias exactamente iguales de un conductor duplicado. La normalización de identificadores y la selección entre registros con nombres distintos permanecen pendientes.


## 1 · Configuración y rutas

En local puede ejecutarse desde cualquier carpeta dentro del repositorio. En Colab, ajusta `RUTA_PROYECTO_DRIVE`. Cada ejecución reemplaza las salidas anteriores.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
VERSION_TRATAMIENTO = "GIAD-M3-S4-HR-drivers-tratamiento-v1"

COLUMNAS_ORIGINALES = [
    "chofer_id", "chofer_nombre", "centro_distribucion",
    "horas_conduccion_mes", "salario_base_bob", "ausentismo_dias",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

def configurar_rutas():
    entorno = ENTORNO
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    n1 = raiz / "S4" / "andinalog_hr_drivers" / "notebook1" / "salidas"
    n2 = raiz / "S4" / "andinalog_hr_drivers" / "notebook2"
    rutas = {
        "bronze": raiz / "datasets" / CARPETA_DATASETS / "andinalog_hr_drivers.csv",
        "diagnosticado": n1 / "andinalog_hr_drivers_diagnosticado.csv",
        "problemas": n1 / "andinalog_hr_drivers_problemas.csv",
        "reporte_n1": n1 / "andinalog_hr_drivers_reporte_calidad.csv",
        "catalogo": n2 / "catalogo_reglas_tratamiento.csv",
        "salidas": n2 / "salidas",
    }
    faltantes = [str(p) for clave,p in rutas.items() if clave != "salidas" and not p.is_file()]
    if faltantes:
        raise FileNotFoundError("Faltan archivos requeridos:\n" + "\n".join(faltantes))
    return rutas

RUTAS = configurar_rutas()
RUTAS


{'bronze': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/datasets/AndinaLog_03B_Bronce/andinalog_hr_drivers.csv'),
 'diagnosticado': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_hr_drivers/notebook1/salidas/andinalog_hr_drivers_diagnosticado.csv'),
 'problemas': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_hr_drivers/notebook1/salidas/andinalog_hr_drivers_problemas.csv'),
 'reporte_n1': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_hr_drivers/notebook1/salidas/andinalog_hr_drivers_reporte_calidad.csv'),
 'catalogo': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_hr_drivers/notebook2/catalogo_reglas_tratamiento.csv'),
 'salidas': WindowsPath('c:/Users/remrodri/Github/practicasNotebookColab/S4/andinalog_hr_drivers/notebook2/salidas')}

## 2 · Carga, trazabilidad y catálogo

Se comprueba que el archivo diagnosticado corresponde al mismo Bronze mediante la huella SHA-256 registrada por el notebook 1.


In [3]:
def cargar_csv(ruta):
    return pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)

bronze = cargar_csv(RUTAS["bronze"])
diagnosticado = cargar_csv(RUTAS["diagnosticado"])
problemas = cargar_csv(RUTAS["problemas"])
reporte_n1 = cargar_csv(RUTAS["reporte_n1"])
catalogo = cargar_csv(RUTAS["catalogo"])

hash_bronze = hashlib.sha256(RUTAS["bronze"].read_bytes()).hexdigest()
hash_reportado = reporte_n1.loc[reporte_n1["metrica"].eq("sha256_bronze"), "valor"].iloc[0]
assert hash_bronze == hash_reportado, "El Bronze no corresponde al diagnóstico del notebook 1"
assert list(bronze.columns) == COLUMNAS_ORIGINALES
assert diagnosticado[COLUMNAS_ORIGINALES].equals(bronze[COLUMNAS_ORIGINALES])
assert catalogo["regla_id"].is_unique
assert catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all()

reglas_aprobadas = set(catalogo.loc[catalogo["estado"].eq("APROBADA"), "regla_id"])
print("Reglas aprobadas:", sorted(reglas_aprobadas))
display(catalogo)


Reglas aprobadas: ['DUPLICADO_IDENTICO']


,regla_id,columna_afectada,codigo_error_o_unidad,tratamiento_propuesto,estado,evidencia_acuerdo,validacion_requerida
0,DUPLICADO_IDENTICO,chofer_id,DUPLICADO,Conservar la primera fila como canónica y excl...,APROBADA,Criterio conservador de duplicados acordado pr...,Mismo chofer_id y las seis columnas originales...
1,NORMALIZAR_CHOFER_ID,chofer_id,FORMATO_INVALIDO,Quitar espacios externos y convertir a mayúsculas,PENDIENTE,,El resultado debe cumplir CHO-### y no crear u...
2,DUPLICADO_NOMBRE_CONFLICTIVO,chofer_id,DUPLICADO,Mantener ambas filas en cuarentena cuando el n...,PENDIENTE,,Fuente autorizada que confirme el nombre correcto
3,CHOFER_ID_FALTANTE,chofer_id,FALTANTE,Mantener en cuarentena sin inventar un identif...,PENDIENTE,,Identificador recuperado de una fuente autorizada
4,NOMBRE_FALTANTE,chofer_nombre,FALTANTE,Mantener en cuarentena sin imputar un nombre,PENDIENTE,,Nombre recuperado de una fuente autorizada
5,NOMBRE_FORMATO_INVALIDO,chofer_nombre,FORMATO_INVALIDO,Mantener en cuarentena hasta validar el nombre,PENDIENTE,,Nombre completo confirmado por una fuente auto...
6,CENTRO_FALTANTE,centro_distribucion,FALTANTE,Mantener en cuarentena sin imputación automática,PENDIENTE,,Centro confirmado por una fuente autorizada
7,CENTRO_NO_RECONOCIDO,centro_distribucion,VALOR_NO_RECONOCIDO,Mantener en cuarentena hasta validar el centro,PENDIENTE,,Centro incluido en el dominio operativo aprobado
8,HORAS_FALTANTES,horas_conduccion_mes,FALTANTE,Mantener en cuarentena sin imputación automática,PENDIENTE,,Horas recuperadas de una fuente autorizada
9,HORAS_NO_NUMERICAS,horas_conduccion_mes,NO_NUMERICO,Mantener en cuarentena hasta recuperar el valo...,PENDIENTE,,Valor numérico verificable


## 3 · Preparación y decisiones sobre duplicados

Las columnas terminadas en `_preparado` forman el área de trabajo. Los campos Bronze permanecen intactos. Una copia exacta se conserva para auditoría, pero se marca como no utilizable.


In [4]:
def preparar_base(df):
    salida = df.copy(deep=True)
    for columna in COLUMNAS_ORIGINALES:
        salida[f"{columna}_preparado"] = salida[columna]
    salida["registro_canonico"] = True
    salida["excluido_como_copia"] = False
    return salida

def decidir_duplicados(df):
    decisiones = []
    original = df[COLUMNAS_ORIGINALES]
    for chofer_id, indices in df.groupby(df["chofer_id"].str.strip(), sort=False).groups.items():
        indices = list(indices)
        if len(indices) < 2:
            continue
        canonica = indices[0]
        for idx in indices[1:]:
            exacta = original.loc[idx].equals(original.loc[canonica])
            if exacta and "DUPLICADO_IDENTICO" in reglas_aprobadas:
                decision, regla = "COPIA_EXCLUIDA", "DUPLICADO_IDENTICO"
                justificacion = "Las seis columnas originales coinciden con la primera fila del conductor"
            else:
                decision, regla = "PENDIENTE", "DUPLICADO_NOMBRE_CONFLICTIVO"
                justificacion = "Las filas presentan diferencias o no existe una regla aprobada para seleccionar una"
            decisiones.append({
                "fila_bronze": int(df.at[idx, "fila_bronze"]),
                "chofer_id": df.at[idx, "chofer_id"],
                "fila_canonica": int(df.at[canonica, "fila_bronze"]),
                "decision": decision, "regla_id": regla, "justificacion": justificacion,
            })
    return pd.DataFrame(decisiones, columns=["fila_bronze","chofer_id","fila_canonica","decision","regla_id","justificacion"])

df_trabajo = preparar_base(diagnosticado)
decisiones_duplicados = decidir_duplicados(df_trabajo)
filas_excluidas = set(decisiones_duplicados.loc[decisiones_duplicados["decision"].eq("COPIA_EXCLUIDA"), "fila_bronze"].astype(int))
df_trabajo["excluido_como_copia"] = df_trabajo["fila_bronze"].astype(int).isin(filas_excluidas)
df_trabajo.loc[df_trabajo["excluido_como_copia"], "registro_canonico"] = False
print("Copias exactas excluidas:", len(filas_excluidas))
display(decisiones_duplicados)


Copias exactas excluidas: 4


,fila_bronze,chofer_id,fila_canonica,decision,regla_id,justificacion
0,152,CHO-003,3,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las seis columnas originales coinciden con la ...
1,153,CHO-056,56,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las seis columnas originales coinciden con la ...
2,154,CHO-119,119,PENDIENTE,DUPLICADO_NOMBRE_CONFLICTIVO,Las filas presentan diferencias o no existe un...
3,156,CHO-126,126,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las seis columnas originales coinciden con la ...
4,155,CHO-141,141,COPIA_EXCLUIDA,DUPLICADO_IDENTICO,Las seis columnas originales coinciden con la ...


## 4 · Evaluación problema por problema

Los problemas sin una regla aprobada permanecen `PENDIENTE`; las copias exactas reciben `EXCLUIDO_COMO_COPIA`.


In [5]:
def evaluar_acciones(problemas, decisiones):
    acciones = problemas.copy()
    mapa = decisiones.set_index("fila_bronze")["decision"] if not decisiones.empty else pd.Series(dtype="string")
    acciones["regla_id"] = ""
    acciones["estado_accion"] = "PENDIENTE"
    acciones["valor_preparado"] = acciones["valor_original"]
    acciones["justificacion"] = "No existe una regla aprobada para modificar este problema"
    es_dup = acciones["codigo_error"].eq("DUPLICADO")
    decision = acciones["fila_bronze"].astype(int).map(mapa).fillna("PENDIENTE")
    excluida = es_dup & decision.eq("COPIA_EXCLUIDA")
    acciones.loc[excluida, "regla_id"] = "DUPLICADO_IDENTICO"
    acciones.loc[excluida, "estado_accion"] = "EXCLUIDO_COMO_COPIA"
    acciones.loc[excluida, "justificacion"] = "Copia exacta retenida para auditoría y excluida del conjunto utilizable"
    acciones["version_tratamiento"] = VERSION_TRATAMIENTO
    return acciones

actions = evaluar_acciones(problemas, decisiones_duplicados)
pendientes = actions.loc[actions["estado_accion"].eq("PENDIENTE")].groupby("fila_bronze")["columna_afectada"].agg(lambda x: "|".join(dict.fromkeys(x)))
df_trabajo["columnas_pendientes"] = df_trabajo["fila_bronze"].map(pendientes).fillna("")
df_trabajo["en_cuarentena_final"] = df_trabajo["columnas_pendientes"].ne("") | df_trabajo["excluido_como_copia"]
df_trabajo["registro_utilizable"] = ~df_trabajo["en_cuarentena_final"] & df_trabajo["registro_canonico"]
df_trabajo["version_tratamiento"] = VERSION_TRATAMIENTO
cuarentena_final = df_trabajo.loc[df_trabajo["en_cuarentena_final"]].copy()
print("Utilizables:", int(df_trabajo["registro_utilizable"].sum()))
print("Cuarentena final:", len(cuarentena_final))


Utilizables: 146
Cuarentena final: 10


## 5 · Validaciones y reporte

Las comprobaciones impiden perder filas, alterar los campos Bronze o admitir identificadores duplicados en el conjunto utilizable.


In [6]:
def construir_reporte():
    datos = [
        ("archivo_bronze", RUTAS["bronze"].name), ("sha256_bronze", hash_bronze),
        ("version_tratamiento", VERSION_TRATAMIENTO), ("filas_totales", len(df_trabajo)),
        ("filas_utilizables", int(df_trabajo["registro_utilizable"].sum())),
        ("filas_cuarentena_final", int(df_trabajo["en_cuarentena_final"].sum())),
        ("copias_exactas_excluidas", int(df_trabajo["excluido_como_copia"].sum())),
        ("problemas_pendientes", int(actions["estado_accion"].eq("PENDIENTE").sum())),
    ]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

reporte = construir_reporte()
pd.testing.assert_frame_equal(df_trabajo[COLUMNAS_ORIGINALES], bronze[COLUMNAS_ORIGINALES])
assert len(df_trabajo) == len(bronze)
assert df_trabajo["fila_bronze"].is_unique
assert set(filas_excluidas).issubset(set(df_trabajo.loc[df_trabajo["en_cuarentena_final"], "fila_bronze"].astype(int)))
assert not df_trabajo.loc[df_trabajo["registro_utilizable"]].duplicated("chofer_id").any()
assert len(cuarentena_final) == int(df_trabajo["en_cuarentena_final"].sum())
display(reporte)
print("Validaciones correctas")


,metrica,valor
0,archivo_bronze,andinalog_hr_drivers.csv
1,sha256_bronze,f4edf8a0f50292e3f320629f26b30a1a725df7e4309654...
2,version_tratamiento,GIAD-M3-S4-HR-drivers-tratamiento-v1
3,filas_totales,156
4,filas_utilizables,146
5,filas_cuarentena_final,10
6,copias_exactas_excluidas,4
7,problemas_pendientes,6


Validaciones correctas


## 6 · Exportación reproducible

Las salidas se escriben primero en archivos temporales y reemplazan versiones anteriores. El Bronze, el diagnóstico y el catálogo nunca se sobrescriben.


In [7]:
def exportar_csvs(directorio, tablas):
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_hr2_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        if hashlib.sha256(RUTAS["bronze"].read_bytes()).hexdigest() != hash_bronze:
            raise RuntimeError("El Bronze cambió durante la ejecución")
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

salidas = exportar_csvs(RUTAS["salidas"], {
    "andinalog_hr_drivers_tratado.csv": df_trabajo,
    "andinalog_hr_drivers_acciones.csv": actions,
    "andinalog_hr_drivers_decisiones_duplicados.csv": decisiones_duplicados,
    "andinalog_hr_drivers_cuarentena_final.csv": cuarentena_final,
    "andinalog_hr_drivers_reporte_tratamiento.csv": reporte,
})
for ruta in salidas:
    print(ruta)


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook2\salidas\andinalog_hr_drivers_tratado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook2\salidas\andinalog_hr_drivers_acciones.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook2\salidas\andinalog_hr_drivers_decisiones_duplicados.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook2\salidas\andinalog_hr_drivers_cuarentena_final.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook2\salidas\andinalog_hr_drivers_reporte_tratamiento.csv


## Revisión posterior

La normalización de `chofer_id` y la decisión sobre duplicados con nombres diferentes deben aprobarse antes de cambiar su estado en el catálogo. Mientras sigan pendientes, el notebook conserva esas filas en cuarentena.
